In [0]:
%sql
-- Check if gold tables were created
SHOW TABLES IN adtech_catalog.gold;

database,tableName,isTemporary
gold,ad_performance_analytics,false
gold,fact_ad_performance,false


In [0]:
-- Count rows in fact_ad_performance
-- Expected: ~1,000 rows (one per ad)
SELECT 
    'fact_ad_performance' as table_name,
    COUNT(*) as row_count
FROM adtech_catalog.gold.fact_ad_performance;

table_name,row_count
fact_ad_performance,1000


In [0]:
-- Count rows by category (distribution check)
SELECT 
    ad_category,
    COUNT(*) as ad_count
FROM adtech_catalog.gold.fact_ad_performance
GROUP BY ad_category
ORDER BY ad_count DESC;

ad_category,ad_count
Electronics,182
Travel,179
Food,175
Health,160
Gaming,148
Fashion,147
electronics,9


In [0]:
-- View all columns in the gold table
DESCRIBE adtech_catalog.gold.fact_ad_performance;

col_name,data_type,comment
Ad_Reference_ID,string,null
ad_category,string,null
ad_device,string,null
ad_location,string,null
ad_type,string,null
ad_type_catalog,string,null
cost_per_click,"decimal(10,2)",null
ad_video_length,double,null
total_clicks,bigint,null
total_impressions,bigint,null


In [0]:
-- Sample 10 rows from gold table
SELECT 
    Ad_Reference_ID,
    ad_category,
    ad_type,
    ad_device,
    cost_per_click,
    total_impressions,
    total_clicks,
    ctr,
    roas,
    conversion_rate,
    high_performance,
    ad_lifecycle_stage,
    engagement_score
FROM adtech_catalog.gold.fact_ad_performance
LIMIT 10;

Ad_Reference_ID,ad_category,ad_type,ad_device,cost_per_click,total_impressions,total_clicks,ctr,roas,conversion_rate,high_performance,ad_lifecycle_stage,engagement_score
AD_86983,Fashion,Video,Mobile,1.20,278,28,0.10071942446043165,0.0,0.0,0,New,0.040287769784172665
AD_79262,Fashion,Image,Desktop,0.25,301,31,0.10299003322259136,1.499835643220934,0.16129032258064516,0,New,0.3579160959855684
AD_65703,Fashion,Text,All-Devices,0.00,271,29,0.1070110701107011,0.0,0.034482758620689655,0,New,0.34722215586015437
AD_99006,Fashion,Video,Mobile,3.46,275,32,0.11636363636363636,0.0,0.0,0,New,0.04654545454545455
AD_12233,Fashion,Text,Tablet,0.00,242,26,0.10743801652892562,0.0,0.0,0,New,0.042975206611570255
AD_78902,Fashion,Image,Tablet,0.88,267,24,0.0898876404494382,1.3267138665239948,0.125,0,New,0.3343116135494143
AD_10784,Fashion,Video,All-Devices,1.18,263,28,0.10646387832699619,0.0,0.0,0,New,0.04258555133079848
AD_17121,Fashion,Image,Desktop,2.98,243,29,0.11934156378600823,0.0,0.0,0,New,0.047736625514403296
AD_76665,Fashion,Video,Desktop,2.65,235,27,0.1148936170212766,0.6959286226737263,0.1111111111111111,0,New,0.34549102164405904
AD_61310,Fashion,Image,All-Devices,2.05,235,28,0.11914893617021277,0.0,0.0,0,New,0.04765957446808511


In [0]:
-- Overall statistics
SELECT 
    COUNT(*) as total_ads,
    ROUND(AVG(ctr) * 100, 2) as avg_ctr_pct,
    ROUND(AVG(roas), 2) as avg_roas,
    ROUND(AVG(conversion_rate) * 100, 2) as avg_conversion_pct,
    SUM(high_performance) as high_performance_ads,
    ROUND(SUM(high_performance) * 100.0 / COUNT(*), 2) as high_performance_pct,
    ROUND(AVG(engagement_score), 2) as avg_engagement_score
FROM adtech_catalog.gold.fact_ad_performance;

total_ads,avg_ctr_pct,avg_roas,avg_conversion_pct,high_performance_ads,high_performance_pct,avg_engagement_score
1000,9.99,0.34,3.59,34,3.40,0.13


In [0]:
-- Category-wise performance
SELECT 
    ad_category,
    COUNT(*) as ad_count,
    ROUND(AVG(ctr) * 100, 2) as avg_ctr_pct,
    ROUND(AVG(roas), 2) as avg_roas,
    ROUND(AVG(conversion_rate) * 100, 2) as avg_conversion_pct,
    SUM(high_performance) as winning_ads,
    ROUND(SUM(high_performance) * 100.0 / COUNT(*), 2) as win_rate_pct,
    ROUND(AVG(engagement_score), 2) as avg_engagement
FROM adtech_catalog.gold.fact_ad_performance
GROUP BY ad_category
ORDER BY avg_roas DESC;

ad_category,ad_count,avg_ctr_pct,avg_roas,avg_conversion_pct,winning_ads,win_rate_pct,avg_engagement
electronics,9,10.11,1.13,11.06,1,11.11,0.35
Health,160,9.95,0.43,4.25,10,6.25,0.14
Fashion,147,10.01,0.4,4.38,5,3.40,0.16
Travel,179,9.98,0.31,3.51,4,2.23,0.13
Electronics,182,10.05,0.3,3.03,6,3.30,0.12
Food,175,9.95,0.3,3.06,4,2.29,0.12
Gaming,148,9.98,0.29,3.05,4,2.70,0.13


In [0]:
-- Ad type performance
SELECT 
    ad_type,
    COUNT(*) as ad_count,
    ROUND(AVG(ctr) * 100, 2) as avg_ctr_pct,
    ROUND(AVG(roas), 2) as avg_roas,
    ROUND(AVG(conversion_rate) * 100, 2) as avg_conversion_pct,
    ROUND(AVG(engagement_score), 2) as avg_engagement
FROM adtech_catalog.gold.fact_ad_performance
GROUP BY ad_type
ORDER BY avg_roas DESC;

ad_type,ad_count,avg_ctr_pct,avg_roas,avg_conversion_pct,avg_engagement
Text,156,10.14,0.42,4.29,0.15
Video,353,9.9,0.36,3.76,0.14
Image,331,10.16,0.31,3.35,0.13
Unknown,160,9.68,0.29,3.04,0.12


In [0]:
-- Ad lifecycle distribution
SELECT 
    ad_lifecycle_stage,
    COUNT(*) as ad_count,
    ROUND(AVG(roas), 2) as avg_roas,
    ROUND(AVG(ctr) * 100, 2) as avg_ctr_pct
FROM adtech_catalog.gold.fact_ad_performance
GROUP BY ad_lifecycle_stage
ORDER BY 
    CASE ad_lifecycle_stage
        WHEN 'New' THEN 1
        WHEN 'Growing' THEN 2
        WHEN 'Mature' THEN 3
        WHEN 'Declining' THEN 4
    END;

ad_lifecycle_stage,ad_count,avg_roas,avg_ctr_pct
New,1000,0.34,9.99


In [0]:
-- Season performance
SELECT 
    season,
    COUNT(*) as ad_count,
    ROUND(AVG(roas), 2) as avg_roas,
    ROUND(AVG(ctr) * 100, 2) as avg_ctr_pct
FROM adtech_catalog.gold.fact_ad_performance
GROUP BY season
ORDER BY avg_roas DESC;

season,ad_count,avg_roas,avg_ctr_pct
Winter,653,0.35,10.05
Spring,347,0.33,9.87


In [0]:
-- Location type performance
SELECT 
    location_type,
    COUNT(*) as ad_count,
    ROUND(AVG(roas), 2) as avg_roas,
    ROUND(AVG(ctr) * 100, 2) as avg_ctr_pct
FROM adtech_catalog.gold.fact_ad_performance
GROUP BY location_type
ORDER BY avg_roas DESC;

location_type,ad_count,avg_roas,avg_ctr_pct
Semi-Urban,198,0.37,9.73
Urban,595,0.34,10.02
Rural,207,0.32,10.16


In [0]:
-- Cost efficiency distribution
SELECT 
    CASE 
        WHEN cost_efficiency_score > 0.4 THEN 'Excellent (>0.4)'
        WHEN cost_efficiency_score > 0.25 THEN 'Good (0.25-0.4)'
        WHEN cost_efficiency_score > 0.15 THEN 'Average (0.15-0.25)'
        ELSE 'Poor (<0.15)'
    END as efficiency_level,
    COUNT(*) as ad_count,
    ROUND(AVG(roas), 2) as avg_roas
FROM adtech_catalog.gold.fact_ad_performance
GROUP BY efficiency_level
ORDER BY avg_roas DESC;

efficiency_level,ad_count,avg_roas
Poor (<0.15),1000,0.34


In [0]:
-- Engagement score distribution
SELECT 
    CASE 
        WHEN engagement_score > 0.7 THEN 'High (>0.7)'
        WHEN engagement_score > 0.5 THEN 'Medium (0.5-0.7)'
        ELSE 'Low (<0.5)'
    END as engagement_level,
    COUNT(*) as ad_count,
    ROUND(AVG(roas), 2) as avg_roas,
    ROUND(AVG(ctr) * 100, 2) as avg_ctr_pct
FROM adtech_catalog.gold.fact_ad_performance
GROUP BY engagement_level
ORDER BY avg_roas DESC;

engagement_level,ad_count,avg_roas,avg_ctr_pct
Low (<0.5),1000,0.34,9.99


In [0]:
-- Check for nulls in critical columns
SELECT 
    SUM(CASE WHEN Ad_Reference_ID IS NULL THEN 1 ELSE 0 END) as null_ad_reference,
    SUM(CASE WHEN ad_category IS NULL THEN 1 ELSE 0 END) as null_category,
    SUM(CASE WHEN ctr IS NULL THEN 1 ELSE 0 END) as null_ctr,
    SUM(CASE WHEN roas IS NULL THEN 1 ELSE 0 END) as null_roas,
    SUM(CASE WHEN conversion_rate IS NULL THEN 1 ELSE 0 END) as null_conversion,
    SUM(CASE WHEN high_performance IS NULL THEN 1 ELSE 0 END) as null_high_performance
FROM adtech_catalog.gold.fact_ad_performance;

null_ad_reference,null_category,null_ctr,null_roas,null_conversion,null_high_performance
0,0,0,0,0,0


In [0]:
-- Check that gold has correct number of ads
-- Should match silver catalog count (1,000)
SELECT 
    'Bronze/Silver Catalog' as source,
    COUNT(*) as ad_count
FROM adtech_catalog.silver.conformed_ad_catalog
UNION ALL
SELECT 
    'Gold Fact' as source,
    COUNT(*) as ad_count
FROM adtech_catalog.gold.fact_ad_performance;

source,ad_count
Bronze/Silver Catalog,1000
Gold Fact,1000


In [0]:
-- Check if Gold run was logged
SELECT 
    version_id,
    deployed_at,
    description,
    status
FROM adtech_catalog.monitoring.version_history
WHERE description LIKE '%Gold%'
ORDER BY deployed_at DESC;

version_id,deployed_at,description,status
20260730_113036,2026-07-30T11:31:16.139583,Gold Layer - Feature Engineering,SUCCESS


In [0]:
-- One-line summary
SELECT 
    'GOLD LAYER' as layer,
    COUNT(*) as total_rows,
    CAST(AVG(roas) AS DECIMAL(5,2)) as avg_roas,
    CAST(AVG(ctr) * 100 AS DECIMAL(5,2)) as avg_ctr_pct,
    SUM(high_performance) as winning_ads,
    'PASSED' as quality_status
FROM adtech_catalog.gold.fact_ad_performance;

layer,total_rows,avg_roas,avg_ctr_pct,winning_ads,quality_status
GOLD LAYER,1000,0.34,9.99,34,PASSED
